# Actividad 4. Selección de modelos y prototipado rápido
## Auditoría de conjuntos de datos para reconocimiento de placas vehiculares
**Continuidad:** Fase I – Business Case – Unidad Habitacional TecMilenio.  
**Objetivo:** verificar integridad, estructura, etiquetas, duplicados y adecuación de fuentes públicas antes de seleccionar un conjunto de evaluación.  
**Glosario:** OCR: *Optical Character Recognition* (reconocimiento óptico de caracteres); CPU: *Central Processing Unit* (unidad central de procesamiento); GPU: *Graphics Processing Unit* (unidad de procesamiento gráfico); CSV: *Comma-Separated Values* (valores separados por comas); SHA-256: *Secure Hash Algorithm 256* (algoritmo criptográfico de resumen de 256 bits).  
**Entorno:** Google Colab, CPU suficiente. **No se realiza entrenamiento ni se ejecutan modelos OCR en esta auditoría.**

### Criterios de calidad y trazabilidad
Se consideran fuentes candidatas *sonnetechnology/license-plate-text-recognition-full* y *prithivMLmods/Number-Plate-Recognition*. Su elección definitiva dependerá de la evidencia obtenida. Las etiquetas automáticas no se considerarán verdad de referencia sin verificación. La coincidencia de texto entre registros tampoco demuestra por sí sola duplicación de imágenes.  
**Nota de alcance:** la auditoría automática identifica posibles errores; la revisión visual y la verificación manual de las transcripciones siguen siendo necesarias. Un conjunto con matrículas de otras regiones no sustituye la evaluación con matrículas mexicanas para un despliegue real.

In [ ]:
# CELDA 1 — Preparación del entorno (CPU)
# Instalar las librerías necesarias para acceder a datasets y analizar imágenes.
%pip -q install -U "datasets>=3.0,<5" "huggingface_hub>=0.28" pillow pandas matplotlib tqdm
print("Instalación terminada. Si Colab solicita reiniciar la sesión, hacerlo antes de continuar.")

In [ ]:
# CELDA 2 — Importaciones y configuración reproducible
import re, io, os, json, time, hashlib, random
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from datasets import load_dataset, get_dataset_config_names
random.seed(42)
DIRECTORIO = Path('/content/auditoria_ocr')
DIRECTORIO.mkdir(parents=True, exist_ok=True)
# Auditar primero una muestra controlada; incrementar solo cuando hayamos validado esquemas y tiempos.
LIMITE_POR_FUENTE = 150
FUENTES = {
    'sonne': 'sonnetechnology/license-plate-text-recognition-full',
    'number_plate': 'prithivMLmods/Number-Plate-Recognition'
}
print('Carpeta de resultados:', DIRECTORIO)
print('Límite de inspección por fuente:', LIMITE_POR_FUENTE)

In [ ]:
# CELDA 3 — Carga controlada y revisión de esquemas
# stream=True evita descargar necesariamente toda la colección; algunos repositorios requieren ajustes.
registros = {}
estados = []
for alias, identificador in FUENTES.items():
    inicio = time.perf_counter()
    try:
        ds = load_dataset(identificador, split='train', streaming=True)
        ejemplos = list(ds.take(LIMITE_POR_FUENTE))
        registros[alias] = ejemplos
        estados.append({'fuente': alias, 'id': identificador, 'registros_cargados':len(ejemplos),
                        'columnas': ', '.join(ejemplos[0].keys()) if ejemplos else 'sin datos',
                        'segundos': round(time.perf_counter()-inicio,2), 'estado': 'OK'})
    except Exception as e:
        registros[alias] = []
        estados.append({'fuente':alias, 'id':identificador, 'registros_cargados':0,
                        'columnas':'', 'segundos':round(time.perf_counter()-inicio,2),
                        'estado':f'{type(e).__name__}: {str(e)[:220]}'})
display(pd.DataFrame(estados))
print('IMPORTANTE: la muestra cargada no equivale al total declarado del dataset.')

In [ ]:
# CELDA 4 — Inspección segura de campos y tipos
# Mostrar solo resumen de campos, sin imprimir imágenes ni exponer datos masivos.
for fuente, filas in registros.items():
    print(f'\n{fuente}: {len(filas)} registros en la muestra')
    for f in filas[:2]:
        print({k: (f'<Imagen PIL {v.size}>' if isinstance(v, Image.Image) else
                   f'<bytes {len(v)}>' if isinstance(v, bytes) else
                   str(v)[:180]) for k,v in f.items()})

In [ ]:
# CELDA 5 — Extracción adaptable de imágenes y anotaciones
# Esquemas distintos requieren tratamiento distinto. No inventamos etiquetas faltantes.
def como_pil(valor):
    try:
        if isinstance(valor, Image.Image): return ImageOps.exif_transpose(valor).convert('RGB')
        if isinstance(valor, dict):
            if valor.get('bytes'): return Image.open(io.BytesIO(valor['bytes'])).convert('RGB')
            if valor.get('path') and os.path.isfile(valor['path']): return Image.open(valor['path']).convert('RGB')
    except Exception: pass
    return None

def primer_texto(row):
    for k in ('license_plate_number','plate','plate_number','text','label','target','transcription'):
        if k in row and row[k] is not None:
            v=row[k]
            if isinstance(v, str): return v, k
            if isinstance(v, list): return ' | '.join(map(str,v)),k
    return '', 'no_identificada'

def obtener_imagen(row):
    for k in ('image','img','picture','image_file'):
        if k in row:
            img=como_pil(row[k])
            if img is not None:return img,k
    return None,'no_identificada'

indice=[]
imagenes={}
for fuente, filas in registros.items():
    for i,r in enumerate(filas):
        im,campo_im=obtener_imagen(r)
        etiqueta,campo_et=primer_texto(r)
        key=f'{fuente}_{i:05d}'
        if im is not None:
            # Hash del contenido visual normalizado: detecta imágenes idénticas aunque cambie el archivo.
            h=hashlib.sha256(im.tobytes()+str(im.size).encode()).hexdigest()
            imagenes[key]=im
            w,hg=im.size
        else: h,w,hg=None,None,None
        indice.append({'id':key,'fuente':fuente,'campo_imagen':campo_im,
                       'campo_etiqueta':campo_et,'texto_original':etiqueta,
                       'ancho':w,'alto':hg,'hash_imagen':h,
                       'imagen_legible':im is not None,
                       'bbox_disponible':any(k in r for k in ('bbox','boxes','bounding_box','annotations'))})
auditoria=pd.DataFrame(indice)
display(auditoria.groupby('fuente',dropna=False).agg(
    registros=('id','count'), imagenes_legibles=('imagen_legible','sum'),
    con_bbox=('bbox_disponible','sum')).reset_index() if len(auditoria) else pd.DataFrame())

In [ ]:
# CELDA 6 — Detección de alertas y duplicados exactos
# Este filtro es exploratorio; patrones de placas válidos varían entre países y entidades.
if len(auditoria):
    auditoria['texto_normalizado']=auditoria['texto_original'].fillna('').astype(str).str.upper().str.strip()
    auditoria['texto_vacio']=auditoria['texto_normalizado'].eq('')
    auditoria['texto_sospechoso']=auditoria['texto_normalizado'].str.contains(
        r'TOTAL|INVOICE|RECEIPT|PRICE|\*{2,}|\||\n',regex=True,na=False)
    auditoria['duplicado_imagen_exacto']=auditoria['hash_imagen'].notna() & auditoria['hash_imagen'].duplicated(keep=False)
    auditoria['etiqueta_repetida']=auditoria['texto_normalizado'].ne('') & auditoria.duplicated(
        subset=['fuente','texto_normalizado'],keep=False)
    auditoria['requiere_revision']=(~auditoria['imagen_legible'] | auditoria['texto_vacio'] |
        auditoria['texto_sospechoso'] | auditoria['duplicado_imagen_exacto'])
    resumen=auditoria.groupby('fuente').agg(
        registros=('id','size'), sin_imagen=('imagen_legible',lambda s:int((~s).sum())),
        texto_vacio=('texto_vacio','sum'), texto_sospechoso=('texto_sospechoso','sum'),
        duplicado_exacto=('duplicado_imagen_exacto','sum'), etiqueta_repetida=('etiqueta_repetida','sum'),
        revision=('requiere_revision','sum')).reset_index()
    display(resumen)
else: print('No hay registros. Revisar los errores de la CELDA 3.')
print('Nota: una etiqueta repetida NO equivale a una imagen duplicada.')

In [ ]:
# CELDA 7 — Muestra visual para revisión humana
# Revisar si la imagen es placa completa, recorte correcto, o fotografía de un vehículo/escena.
if imagenes:
    ids=list(imagenes.keys())
    seleccion=random.Random(42).sample(ids,min(16,len(ids)))
    filas=(len(seleccion)+3)//4
    fig,ax=plt.subplots(filas,4,figsize=(16,3.6*filas))
    for a in np.array(ax).reshape(-1): a.axis('off')
    for a,id_ in zip(np.array(ax).reshape(-1),seleccion):
        a.imshow(imagenes[id_]);a.axis('off')
        texto=auditoria.loc[auditoria.id.eq(id_),'texto_original'].iloc[0]
        a.set_title(f'{id_}\nEtiqueta: {str(texto)[:35]}',fontsize=9)
    plt.tight_layout();plt.show()
else: print('No hay imágenes para visualizar.')

In [ ]:
# CELDA 8 — Plantilla de verificación manual
# No se asignará automáticamente 'correcto': esa decisión exige contrastar la imagen con el texto.
if len(auditoria):
    revision=auditoria[['id','fuente','texto_original','imagen_legible','texto_sospechoso',
                        'duplicado_imagen_exacto','requiere_revision']].copy()
    revision['es_placa_visible']='PENDIENTE'  # SI / NO
    revision['placa_recortada']='PENDIENTE' # SI / NO
    revision['transcripcion_verificada']=''  # introducir caracteres visibles
    revision['aprobado_para_evaluacion']='PENDIENTE' # SI / NO
    revision['motivo_exclusion']=''
    ruta=DIRECTORIO/'revision_manual_plantilla.csv'
    revision.to_csv(ruta,index=False,encoding='utf-8-sig')
    print('Plantilla creada:',ruta)
    display(revision.head(10))
else: print('No se generó plantilla porque no hay registros.')

In [ ]:
# CELDA 9 — Exportación de la auditoría preliminar
# Solo resultados efectivamente observados: NO equivale a dataset final aprobado.
if len(auditoria):
    ruta_a=DIRECTORIO/'auditoria_preliminar.csv'
    ruta_r=DIRECTORIO/'resumen_auditoria.csv'
    auditoria.drop(columns=['hash_imagen']).to_csv(ruta_a,index=False,encoding='utf-8-sig')
    resumen.to_csv(ruta_r,index=False,encoding='utf-8-sig')
    print('Archivos exportados:',ruta_a,ruta_r,sep='\n - ')
    print('Cantidad inspeccionada:',len(auditoria),'(no extrapolar tasas al total sin muestreo representativo)')
else: print('Sin resultados que exportar.')

### Dictamen y siguientes decisiones
**Estado por defecto: decisión pendiente.** Antes de seleccionar definitivamente un dataset, verificar imágenes, coordenadas y etiquetas; determinar si hay matrículas duplicadas; documentar descartes. Para el experimento final se requiere una muestra de placas recortadas con transcripciones humanas verificadas y una política clara de exclusión.  
**Separación metodológica:** este notebook no ejecuta reconocimiento OCR ni calcula rendimiento de modelos. La comparación de los tres modelos se desarrollará en otro notebook y con un conjunto de prueba común.  
**Fuentes:** [Sonne Technology](https://huggingface.co/datasets/sonnetechnology/license-plate-text-recognition-full) y [Number Plate Recognition](https://huggingface.co/datasets/prithivMLmods/Number-Plate-Recognition). Revalidar sus condiciones de uso antes de publicar imágenes en GitHub.  
**Limitación:** la ejecución real en Google Colab es necesaria para confirmar estructura, disponibilidad y calidad; ningún valor de salida del código se considera observado hasta ejecutar las celdas.